In [ ]:
import os

from utils.general import print_gpu_info

os.environ["CUDA_VISIBLE_DEVICES"] = "0"
print_gpu_info()

In [ ]:
from llm_attribution.LLMAnalyzer import LLMAnalyzer
from llm_attribution.utils_attribution import AttributionMethod
from utils.custom_chat_template import custom_apply_chat_template


model_id = "meta-llama/Meta-Llama-3.1-8B-Instruct"
analyzer = LLMAnalyzer(model_id=model_id, device="auto")
messages = []
user_prompt_decision = """I am feeling nervous about my midterm tomorrow. I fear that
Choose the most plausible answer, respond only with the answer and the description:
A) the professor will delay the midterm.
B) I will doodle on my exam and receive points for it.
C) my grandpa has diabetes.
D) I will fail.
"""
messages.append({"role": "user", "content": user_prompt_decision})
decision_prompt = custom_apply_chat_template(messages)
generated_output = analyzer.generate_output(decision_prompt)
print(f"Generated output Check: {generated_output}")
method_params = {
    AttributionMethod.LIME: {
        "n_samples": 500,
        # "baselines": analyzer.tokenizer.pad_token_id,
        "perturbations_per_eval": 500,
    },
    AttributionMethod.SHAPLEY_VALUE_SAMPLING: {
        "n_samples": 100,
        # "baselines": analyzer.tokenizer.pad_token_id,
        "perturbations_per_eval": 100,
    },
    AttributionMethod.LIG: {
        "n_steps": 60,
    },
    AttributionMethod.KERNEL_SHAP: {
        "n_samples": 50,
        # "baselines": analyzer.tokenizer.pad_token_id,
        "perturbations_per_eval": 50,
    },
    AttributionMethod.FEATURE_ABLATION: {
        "perturbations_per_eval": 100,
    },
}

In [ ]:
ablation_results = analyzer.analyze_feature_ablation(
    input_text=decision_prompt,
    target=generated_output,
    **method_params[AttributionMethod.FEATURE_ABLATION],
)
sorted_all_tokens = sorted(
ablation_results.seq_attr_dict.items(), key=lambda x: abs(x[1]), reverse=True
)[:15]
print("All tokens (sorted by absolute value):")
for token, value in sorted_all_tokens:
    print(f"{token}: {value}")

In [ ]:
shapley_results = analyzer.analyze_kernel_shap(
    input_text=decision_prompt,
    target=generated_output,
    **method_params[AttributionMethod.KERNEL_SHAP],
)
sorted_all_tokens = sorted(
    shapley_results.seq_attr_dict.items(), key=lambda x: abs(x[1]), reverse=True
)[:15]
print("All tokens (sorted by absolute value):")
for token, value in sorted_all_tokens:
    print(f"{token}: {value}")

In [ ]:
# LIG Method
lig_results = analyzer.analyze_layer_integrated_gradients(
    input_text=decision_prompt,
    target=generated_output,
    # static_texts=[
    #     "Choose the most plausible answer:\n",
    #     "Respond only with 'A', 'B', 'C', or 'D'.",
    # ],
    **method_params[AttributionMethod.LIG],
)
sorted_all_tokens = sorted(
    lig_results.seq_attr_dict.items(), key=lambda x: abs(x[1]), reverse=True
)[:15]
print("All tokens (sorted by absolute value):")
for token, value in sorted_all_tokens:
    print(f"{token}: {value}")

In [ ]:
# LIME Method
lime_results = analyzer.analyze_lime(
    input_text=decision_prompt,
    target=generated_output,
    # static_texts=["Choose the most plausible answer:\n",
    #                 "Respond only with 'A', 'B', 'C', or 'D'."],
    **method_params[AttributionMethod.LIME]
)
sorted_all_tokens = sorted(
    lime_results.seq_attr_dict.items(),
    key=lambda x: abs(x[1]),
    reverse=True
)[:15]
print("All tokens (sorted by absolute value):")
for token, value in sorted_all_tokens:
    print(f"{token}: {value}")

In [ ]:
shapley_results = analyzer.analyze_shapley_value_sampling(
    input_text=decision_prompt,
    target=generated_output,
    **method_params[AttributionMethod.SHAPLEY_VALUE_SAMPLING],
)
sorted_all_tokens = sorted(
    shapley_results.seq_attr_dict.items(), key=lambda x: abs(x[1]), reverse=True
)[:15]
print("All tokens (sorted by absolute value):")
for token, value in sorted_all_tokens:
    print(f"{token}: {value}")